In this notebook, we will apply Z3 Constraint Solvers to chemistry by calculating the proper distribution of electrons to draw valid Lewis Structures.

**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver, draw_lewis_from_model
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Integers

Let's use Z3 to solve problems involving integers. Let's start with something simple: find $x$ such that

$$2x + 5 = 15$$

In [ ]:
# Initialize variables

x = Int('x') # declairing that x is an integer named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 15 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= 5$$

If the solver returns "**unsat**" then $x=5$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5 ) # REPLACE THIS LINE
s.check()

## Drawing Lewis Structures using Z3

Now that we have familiarized ourselves with Z3, let's see how we can use it to draw the Lewis Structure of a molecule.
The molecule we will start with is $CO_2$. 

To draw the Lewis Structure, we must first determine the number of valence electrons. Carbon has 4, and each oxygen has 6, giving a total of 16.
Now, we have to determine how these electrons will be distributed as bond pairs or lone pairs.

In this molecule, carbon is the central atom, so the bonds will be between the carbon atom and the two oxygen atoms. 
We can represent the number of bond and lone pairs for each atom as variables.


In [ ]:
# Initialize Solver
s = Solver()

# Set the total number of valence electrons
total_valence = 16

# Number of Lone Pairs
C = Int('C')
O1 = Int('O^1')
O2 = Int('O^2')

# Number of Bond Pairs
C_O1 = Int('CO^1')
C_O2 = Int('CO^2')

Great! Our next step is to add the constraints.

1. **Total Electrons:** To create the Lewis structure, all valence electrons must be distributed. So, the total number of pairs multiplied by two should equal our total number of electrons.
2. **Octet Rule:** Each atom must have a full shell of 8 valence electrons. In other words, each element must have 4 electron pairs in total (bonds + lone pairs).
3. **Connectivity:** We know that carbon and oxygen must be bonded, so there is at least a single bond between those atoms.
4. **Non-Negativity:** A molecule cannot have negative electrons! All lone and bond pairs must be $\ge 0$.

**Replace the lines below** to encode the constraints using Z3.

In [ ]:
# 1) The number of electrons distributed must equal 16
s.add((O1 + 1) * 2 == total_valence) # REPLACE THIS LINE

# 2) The octet rule: Each atom must have 4 pairs in total, whether bond or lone pairs.
s.add(O1 + C_O1 == 4)   # All pairs of the first oxygen atom
s.add(O2 == 4)          # REPLACE THIS LINE
s.add(False)            # REPLACE THIS LINE

# 3) Each bond must be at least a single bond
s.add(C_O1 >= 1)
s.add(False) # REPLACE THIS LINE

# 4) Non-negativity: Pairs cannot be less than zero
s.add(C >= 0, O1 >= 0, O2 >= 0, C_O1 >= 0, C_O2 >= 0)

# Let's view the solver 
showSolver(s)

Finally, there is one last constraint we need to add. 

Remember, some molecules have multiple valid structures, or resonance structures.
The most significant resonance structure has a minimum formal charge. If possible, the number of lone pairs for the outer oxygen atoms should be equal.

We can accomplish this using the Z3 `Optimize()` solver to minimize the difference between them, as shown below.

In [ ]:
opt = Optimize()
opt.add(s.assertions())

# We use Z3's If-condition to calculate absolute difference: |O2 - O1|
diff = If(O2 >= O1, O2 - O1, O1 - O2)
opt.minimize(diff)

Now that we've created the solver, all that's left is to view the solution!

In [ ]:
print(opt.check()) # check if solution exists

In [ ]:
print(opt.model()) # output solution

To better view the solution, we've defined a function to draw the structure. 

> **Note:** Bond angles are not physically accurate here. This is only meant to display valid topological electron placement.

In [ ]:
m = opt.model()
draw_lewis_from_model(m)

## Lewis Structure of Nitrogen Triiodide

Great work! Next, let's try a slightly larger molecule, $NI_3$.

In this molecule, nitrogen is the central atom, bonded to three iodine atoms.

**Replace lines in the code below** to create the solver for this molecule.

In [ ]:
# Initialize Solver
s = Solver()

# Set the total number of valence electrons
total_valence = 0 # REPLACE THIS LINE

# Number of Lone Pairs
N = Int('N')
I1 = Int('I^1')
I2 = Int('I^2')
I3 = Int('I^3')

# Number of Bond Pairs
N_I1 = Int('NI^1')
N_I2 = Int('NI^2')
N_I3 = Int('NI^3')

# 1) The number of electrons distributed must equal the total valence
s.add((I1) * 2 == total_valence) # REPLACE THIS LINE

# 2) The octet rule: Each atom must have 4 pairs in total, whether bond or lone pairs.
s.add(False) # REPLACE THIS LINE
s.add(False) # REPLACE THIS LINE
s.add(False) # REPLACE THIS LINE
s.add(False) # REPLACE THIS LINE

# 3) Each bond must be at least a single bond
s.add(False) # REPLACE THIS LINE
s.add(False) # REPLACE THIS LINE
s.add(False) # REPLACE THIS LINE

# 4) Non-negativity
s.add(N >= 0, I1 >= 0, I2 >= 0, I3 >= 0, N_I1 >= 0, N_I2 >= 0, N_I3 >= 0)

# Let's view the solver 
showSolver(s)

In [ ]:
print(s.check()) # check if solution exists

In [ ]:
print(s.model()) # output solution

When you think your solution is correct, use the cell below to visualize it.

In [ ]:
m = s.model()
draw_lewis_from_model(m)



####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm